In [1]:
# worker : Master로부터 신호를 받아 FFT 계산 후 결과를 다시 전송

In [3]:
import socket
import numpy as np

# 서버 설정
HOST = "localhost" #마스터 서버 주소
PORT = 9995  # 통신 포트

# 데이터 설정
signal_length = 30000
expected_bytes = signal_length *4 # 마스터와 데이터를 맞춰야한다는데... 검색해서 더 알아봐야 할듯?

# 소켓 생성
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s: # session 생성
    print("worker : master 연결 시도 중 ...")

    #master 접속
    s.connect((HOST, PORT))
    print("worker : 연결 성공!")

    while True:
        #  데이터 수신
        received = b""

        while len(received) < expected_bytes:
            chunk = s.recv(4096)

            if not chunk:
                print("worker : 연결 종료")
                break

            received += chunk
        if len(received) == 0:
            break

        # 데이터 변환
        signal = np.frombuffer(received, dtype = np.float32) # 마스터에게 받아온 바이터 데이터를 다시 float 타입으로 변환
        print(f"worker : 신호 수신 완료 ( {len(signal)}개)")

        # FFT 계산( 복소수 -> 절대값(진폭))
        fft_result = np.abs(np.fft.fft(signal))

        # 결과 직렬화
        fft_bytes = fft_result.astype(np.float32).tobytes()

        # 분석 결과 전송
        s.sendall(fft_bytes)

        print("worker: 결과 전송 완료")

        
        

        

worker : master 연결 시도 중 ...
worker : 연결 성공!
worker : 신호 수신 완료 ( 30000개)
worker: 결과 전송 완료
worker : 연결 종료
